# PathLearn Gaze Demo (BCC, 6 Segments)

This notebook renders a 6x3 comparison grid directly in notebook output:
- raw_gaze
- IDT-IVT results
- boundary


In [ ]:
from pathlib import Path
import sys
import tempfile
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / 'processors').exists() and (cand / 'README.md').exists():
            return cand
    raise RuntimeError('Could not locate repo root from current working directory')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from processors.gaze.GazeData import GazeData

DATASET_ROOT = REPO_ROOT / 'history/case_005'
raw_files = sorted(DATASET_ROOT.glob('seg_*_raw.json'))
print(f'Segments found: {len(raw_files)}')
for p in raw_files:
    print('-', p.name)


In [ ]:
# Render 6 rows x 3 cols: raw_gaze | IDT-IVT results | boundary
import json
import textwrap

if len(raw_files) != 6:
    raise ValueError(f'Expected 6 segments, got {len(raw_files)}')

rows = []
with tempfile.TemporaryDirectory() as td:
    td_path = Path(td)
    for raw_path in raw_files:
        union_path = raw_path.with_name(raw_path.name.replace('_raw.json', '_union.png'))
        payload = json.loads(raw_path.read_text(encoding='utf-8'))
        seg_text = (payload.get('text') or '').strip()
        result = GazeData.export_segment_raw_ivt_idt_comparison(
            raw_path,
            output_dir=td_path,
            union_image_path=union_path,
            attention_tier='mid',
            canvas_crop_insets=None,
            point_radius=6,
            boundary_width=6,
        )
        seg_name = raw_path.stem.replace('_raw', '')
        rows.append((seg_name, seg_text, result))

    fig, axes = plt.subplots(6, 3, figsize=(16, 26))
    col_titles = ['raw_gaze', 'IDT-IVT results', 'boundary']

    for c, title in enumerate(col_titles):
        axes[0, c].set_title(title, fontsize=12)

    for r, (seg_name, seg_text, result) in enumerate(rows):
        img_paths = [
            result['output_paths']['initial_gaze'],
            result['output_paths']['processed_gaze'],
            result['output_paths']['processed_boundary'],
        ]
        for c, img_path in enumerate(img_paths):
            axes[r, c].imshow(mpimg.imread(img_path))
            axes[r, c].axis('off')

        wrapped = textwrap.fill(seg_text, width=48) if seg_text else '(no text)'
        left_label = f"{seg_name}\npts {result['raw_point_count']}->{result['processed_point_count']}\n{wrapped}"
        axes[r, 0].text(
            -0.03,
            0.5,
            left_label,
            transform=axes[r, 0].transAxes,
            ha='right',
            va='center',
            fontsize=9,
        )

    fig.tight_layout()
    plt.show()
